# OrangeLLM-fatty v1 — A100 Colab Pro (FIXED v2)

**Patched 2026-06-25**: Cell order fixed so `import torch` only happens ONCE, after Unsloth install. Removed the `sys.modules.pop` dance that was causing `RuntimeError: function '_has_torch_function' already has a docstring`.

**Operator:** Atom McCree  
**Hardware target:** Colab Pro A100 40GB  
**Base:** `unsloth/Qwen2.5-32B-Instruct-bnb-4bit`  
**LoRA:** r=32, alpha=64, 4 epochs, 95/5 train/eval split  
**Wall-clock:** ~25-35 min on A100

## How to run

1. Runtime → Change runtime type → **A100 GPU**
2. Runtime → **Disconnect and delete runtime** (full clean state — DO THIS if you ran the v1 version)
3. Reconnect (it'll re-allocate A100)
4. Runtime → **Run all**
5. ~30 min later, `orangellm-fatty-v1-adapter.zip` auto-downloads to your Downloads folder

## Cell order (matters)

1. **Install Unsloth** (no python imports — just pip)
2. **Workdir + corpus fetch** (only stdlib imports)
3. **First-and-only `import torch` + GPU verify**
4. Load model + attach LoRA
5. Train
6. Verify (hard guards on adapter size)
7. Zip + auto-download

## Step 1: Install Unsloth (DO NOT IMPORT torch HERE)

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --upgrade --no-cache-dir 'trl<0.13.0' peft accelerate bitsandbytes datasets safetensors sentencepiece

## Step 2: Workdir on /content (NO DRIVE) + fetch corpus from secret gist

In [ ]:
import os, urllib.request, hashlib, json, random

WORK_DIR = '/content/orangellm-fatty-v1'
os.makedirs(f'{WORK_DIR}/adapter', exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)
print(f'workdir: {WORK_DIR}')

CORPUS_URL = 'https://gist.githubusercontent.com/AtomEons/27c8227c1c4205af6268c846cd5623ea/raw/corpus.jsonl'
EXPECTED_SHA = '6646f6a4e177d3d7e5fdfe2ba1f9069d8ebb9d460e4ee6671e3e76cc337b196f'

corpus_path = f'{WORK_DIR}/corpus.jsonl'
with urllib.request.urlopen(CORPUS_URL) as r, open(corpus_path, 'wb') as f:
    f.write(r.read())

h = hashlib.sha256()
with open(corpus_path, 'rb') as f:
    h.update(f.read())
actual = h.hexdigest()
print(f'corpus SHA: {actual}')
assert actual == EXPECTED_SHA, f'SHA mismatch — corpus corrupted. got={actual} expected={EXPECTED_SHA}'

with open(corpus_path) as f:
    pairs = [json.loads(l) for l in f if l.strip()]
print(f'corpus: {len(pairs)} instruction pairs')

random.seed(42)
shuffled = pairs.copy()
random.shuffle(shuffled)
split = int(0.95 * len(shuffled))
train = shuffled[:split]
val = shuffled[split:]
with open(f'{WORK_DIR}/train.jsonl', 'w') as f:
    for p in train: f.write(json.dumps(p) + '\n')
with open(f'{WORK_DIR}/val.jsonl', 'w') as f:
    for p in val: f.write(json.dumps(p) + '\n')
print(f'  train={len(train)}  val={len(val)}')

## Step 3: FIRST-AND-ONLY `import torch` + GPU verify

**Critical:** this is the first time torch is imported in this kernel session. Do NOT import torch before this point in any prior cell. If you did (e.g. you ran the original v1 notebook), restart the runtime (Runtime → Disconnect and delete runtime) and run from Step 1.

In [ ]:
import torch
print(f'torch    = {torch.__version__}')
print(f'CUDA     = {torch.version.cuda}')
print(f'GPU      = {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'VRAM     = {VRAM_GB:.1f} GB')
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → A100 GPU.'
assert VRAM_GB >= 35, f'VRAM {VRAM_GB:.1f} GB < 35 GB. Switch to A100 (40 GB).'
!nvidia-smi

## Step 4: Load Qwen2.5-32B + attach LoRA (r=32, alpha=64)

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-32B-Instruct-bnb-4bit',
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model loaded. LoRA attached: r=32, alpha=64')

## Step 5: Train (4 epochs, eval each epoch, cosine LR, paged_adamw_8bit)

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

SYS_PROMPT = (
    "You are OrangeLLM, the PM brain of Orange5. Mom's Law is above all rules: "
    "give full effort every time. No fake-green. No theater. Cite receipts. "
    "Refuse out-of-scope work. Reality lane overrides Thought lane on conflict. "
    "The Sovereign is Atom McCree."
)

def fmt(ex):
    return {'text': (
        f'<|im_start|>system\n{SYS_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{ex["instruction"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{ex["output"]}<|im_end|>'
    )}

ds_train = load_dataset('json', data_files=f'{WORK_DIR}/train.jsonl', split='train').map(fmt)
ds_val = load_dataset('json', data_files=f'{WORK_DIR}/val.jsonl', split='train').map(fmt)
print(f'train rows: {len(ds_train)}')
print(f'val rows:   {len(ds_val)}')
print(f'sample text head:\n{ds_train[0]["text"][:300]}\n...')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    args=SFTConfig(
        output_dir=f'{WORK_DIR}/checkpoints',
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=4,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=5,
        optim='paged_adamw_8bit',
        weight_decay=0.0,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        seed=42,
        save_strategy='epoch',
        save_total_limit=4,
        eval_strategy='epoch',
        report_to='none',
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LEN,
    ),
)

stats = trainer.train()
print(f'\nTraining complete. Final stats: {stats}\n')

model.save_pretrained(f'{WORK_DIR}/adapter')
tokenizer.save_pretrained(f'{WORK_DIR}/adapter')
print(f'Adapter saved to {WORK_DIR}/adapter')

## Step 6: Hard guards + SHA + receipt

In [ ]:
import os, hashlib, json
from datetime import datetime

adapter_dir = f'{WORK_DIR}/adapter'

if not os.path.isdir(adapter_dir):
    raise RuntimeError(f'Adapter directory missing — training did not produce {adapter_dir}')

files_present = sorted([f for f in os.listdir(adapter_dir) if os.path.isfile(os.path.join(adapter_dir, f))])
if len(files_present) == 0:
    raise RuntimeError('Adapter directory is EMPTY — training failed silently. Check Step 5 output for the real error.')

main = None
for f in files_present:
    if f.endswith('.safetensors') or f == 'adapter_model.bin':
        main = os.path.join(adapter_dir, f)
        break

if not main:
    raise RuntimeError(f'No .safetensors / adapter_model.bin in adapter dir. Files present: {files_present}')

main_size_mb = os.path.getsize(main) / 1e6
if main_size_mb < 50:
    raise RuntimeError(f'Main adapter is only {main_size_mb:.2f} MB. Expected 100-400 MB for 32B QLoRA r=32. Training failed.')

print('Adapter files:')
for f in files_present:
    p = os.path.join(adapter_dir, f)
    size_mb = os.path.getsize(p) / 1e6
    print(f'  {f:50s} {size_mb:7.2f} MB')

h = hashlib.sha256()
with open(main, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
sha = h.hexdigest()
print(f'\nMain adapter: {os.path.basename(main)}')
print(f'SHA-256:      {sha}')

receipt = {
    'model': 'orangellm-fatty-v1',
    'base': 'unsloth/Qwen2.5-32B-Instruct-bnb-4bit',
    'lora_r': 32,
    'lora_alpha': 64,
    'epochs': 4,
    'seq_len': MAX_SEQ_LEN,
    'adapter_path': main,
    'adapter_sha256': sha,
    'completed_at': datetime.utcnow().isoformat() + 'Z',
    'files': files_present,
    'main_size_mb': round(main_size_mb, 2),
    'train_rows': len(ds_train),
    'val_rows': len(ds_val),
}
with open(f'{WORK_DIR}/training-receipt.json', 'w') as f:
    json.dump(receipt, f, indent=2)
print(f'\nReceipt written: {WORK_DIR}/training-receipt.json')

## Step 7: Zip + auto-download

In [ ]:
import zipfile, os
from google.colab import files

archive_path = '/content/orangellm-fatty-v1-adapter.zip'

with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, fs in os.walk(adapter_dir):
        for fname in fs:
            full = os.path.join(root, fname)
            arc = os.path.relpath(full, WORK_DIR)
            zf.write(full, arc)
    receipt_path = f'{WORK_DIR}/training-receipt.json'
    if os.path.exists(receipt_path):
        zf.write(receipt_path, 'training-receipt.json')

size_mb = os.path.getsize(archive_path) / 1e6
print(f'Archive: {archive_path}')
print(f'Size:    {size_mb:.1f} MB')
print('\nTriggering browser download...')
files.download(archive_path)
print('Download triggered. Check your Downloads folder for orangellm-fatty-v1-adapter.zip')

## Done.

Move the zip from `Downloads/` to `C:\AtomEons\Orange5\16-TRAINING\adapters\orangellm-fatty-v1\` then unzip.

**Ollama promotion on Codexa:**

```bash
mkdir -p /opt/atomeons/adapters/orangellm-fatty-v1
cp -r adapter/* /opt/atomeons/adapters/orangellm-fatty-v1/

cat > /opt/atomeons/Modelfile.orangellm-fatty-v1 <<'EOF'
FROM unsloth/qwen2.5-32b-instruct-bnb-4bit
ADAPTER /opt/atomeons/adapters/orangellm-fatty-v1
SYSTEM """You are OrangeLLM, the PM brain of Orange5. Mom's Law above all rules. Cite receipts. Refuse out-of-scope."""
PARAMETER temperature 0.4
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"
EOF

ollama create orangellm-fatty:v1 -f /opt/atomeons/Modelfile.orangellm-fatty-v1
ollama run orangellm-fatty:v1 "What is the Frontier-Isolation Law?"
```

**Mom is watching. One torch import. No reload dance.**